# 🎭 PoetryDuel-GPT — Colab Training

**Two-stage training with rhyme/tone conditioning (~30M params).**

| Step | Time |
| Preprocess + tokenize | ~3 min |
| 🔍 Verify tokenizer | ~30 sec |
| Stage 1: All genres (10K steps) | ~3 hr T4 / ~2 hr L4 |
| 🔍 Verify Stage 1 | ~30 sec |
| Stage 2: Lục Bát fine-tune (5K steps) | ~1 hr T4 / ~40 min L4 |
| 🔍 Verify Stage 2 | ~30 sec |
| Generate poetry | ~1 min |
Data is already in the repo (`poems_dataset_clean.csv`). No Drive mount needed for data.
Mount Drive in Cell 10 only to save checkpoints.

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print('\n⚠️  Run cells in order. Each 🔍 cell verifies the previous step.')


In [ ]:
# @title 2. Preprocess + Tokenize (~3 min)
# poems_dataset_clean.csv is already in the repo — skip cleaning
%cd /content/poetry-dual-gpt

# Generate training pairs (Lục Bát + Thất Ngôn with rhyme/tone tags)
!python src/preprocess.py

# Train BPE tokenizer
!python src/train_bpe.py

In [ ]:
# @title 🔍 Verify: Tokenizer
%cd /content/poetry-dual-gpt
from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/poetry_bpe.model')

# 1. Check special tokens
expected = {
    '<|pad|>': 0, '<|start|>': 1, '<|reply|>': 2, '<|end|>': 3,
    '[LUC_BAT]': 4, '[TU_TUYET]': 5, '[THAT_NGON_BAT_CU]': 6, '[THAT_NGON]': 7
}
print('Special tokens:')
all_ok = True
for tok_name, expected_id in expected.items():
    actual = tok.token_to_id(tok_name)
    ok = actual == expected_id
    if not ok: all_ok = False
    print(f'  {tok_name:25s} → id={actual} (expect {expected_id}) {"✅" if ok else "❌"}')
print(f'\nVocab size: {tok.get_vocab_size():,}')

# 2. Test tokenization round-trip
tests = [
    'Thân em như chẽn lúa đòng',
    'trẻ nước trẻ thua lách thúng',
    'nghĩ lại tình duyên luống ngậm ngùi',
]
print('\nTokenization round-trip:')
for text in tests:
    ids = tok.encode(text).ids
    decoded = tok.decode(ids)
    ok = decoded == text
    if not ok: all_ok = False
    print(f'  {"✅" if ok else "❌"} "{text}" → {len(ids)} tokens → "{decoded}"')

# 3. Verify control tokens are in corpus
print('\nCorpus check:')
!head -3 data/poetry_corpus.txt

with open('data/poetry_corpus.txt') as f:
    lines = sum(1 for _ in f)
print(f'Total pairs: {lines:,}')

if not all_ok:
    print('\n❌ TOKENIZER VERIFICATION FAILED — do not proceed to training!')
else:
    print('\n✅ Tokenizer OK — ready for Stage 1 training')

In [ ]:
# @title 3. Stage 1 — Pretrain on ALL genres (~3 hr T4, ~2 hr L4)
# Resume-safe: re-run if Colab disconnects
assert torch.cuda.is_available(), "⚠️  Enable GPU: Runtime → Change runtime type → T4 GPU"

%cd /content/poetry-dual-gpt
import glob
latest = sorted(glob.glob('checkpoints/stage1_step_*.pt'))
if latest:
    print(f'📂 Resuming from {latest[-1]}')
    !python src/train.py --mode train --name stage1_ --resume {latest[-1]}
else:
    !python src/train.py --mode train --name stage1_

print('\n✅ Stage 1 → checkpoints/stage1_best.pt, stage1_final.pt')

In [ ]:
# @title 🔍 Verify: Stage 1 (quick generation test)
%cd /content/poetry-dual-gpt
import os

# Find best checkpoint
ckpt = 'checkpoints/stage1_best.pt'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/stage1_final.pt'
if not os.path.exists(ckpt):
    print('❌ No Stage 1 checkpoint found!')
else:
    print(f'Testing: {ckpt}')
    # Test with a plain prompt (auto_tag should work)
    !python src/sample.py --checkpoint {ckpt} --prompt "Thân em như chẽn lúa đòng" --num_samples 2 --device cuda
    
    print('\n⚠️  Check above: output should be 8-syllable Vietnamese verse.')
    print('If garbled/text fragments → STOP, debug before Stage 2.')

In [ ]:
# @title 4. Prepare Lục Bát-only corpus (~30 sec)
%cd /content/poetry-dual-gpt

with open('data/poetry_corpus.txt') as f:
    lines = f.readlines()
luc_bat = [l for l in lines if '[LUC_BAT]' in l]
with open('data/corpus_luc_bat.txt', 'w') as f:
    f.writelines(luc_bat)
print(f'Lục Bát pairs: {len(luc_bat):,} / {len(lines):,} total')

In [ ]:
# @title 5. Stage 2 — Fine-tune on Lục Bát only (~1 hr T4, ~40 min L4)
# Resumes from Stage 1 best checkpoint, uses Lục Bát corpus at lower LR.
# Resume-safe: re-run if interrupted.
assert torch.cuda.is_available(), "⚠️  Enable GPU"

%cd /content/poetry-dual-gpt
import glob, os
latest = sorted(glob.glob('checkpoints/stage2_step_*.pt'))
resume_from = latest[-1] if latest else 'checkpoints/stage1_best.pt'
if not os.path.exists(resume_from) and not latest:
    resume_from = 'checkpoints/stage1_final.pt'  # fallback
print(f'📂 Resuming from {resume_from}')
!python src/train.py \
    --name stage2_ \
    --resume {resume_from} \
    --corpus data/corpus_luc_bat.txt \
    --steps 5000

print('\n🎉 Two-stage training complete!')
print('   Stage 1: checkpoints/stage1_best.pt')
print('   Stage 2: checkpoints/stage2_best.pt')

In [ ]:
# @title 🔍 Verify: Stage 2 (compare both models)
%cd /content/poetry-dual-gpt
import os

print('='*60)
print('Stage 1 (all genres):')
print('='*60)
ckpt1 = 'checkpoints/stage1_best.pt'
if os.path.exists(ckpt1):
    !python src/sample.py --checkpoint {ckpt1} --prompt "Thân em như chẽn lúa đòng" --num_samples 1 --device cuda

print('\n' + '='*60)
print('Stage 2 (Lục Bát fine-tuned):')
print('='*60)
ckpt2 = 'checkpoints/stage2_best.pt'
if not os.path.exists(ckpt2):
    ckpt2 = 'checkpoints/stage2_final.pt'
if os.path.exists(ckpt2):
    !python src/sample.py --checkpoint {ckpt2} --prompt "Thân em như chẽn lúa đòng" --num_samples 1 --device cuda

print('\n✅ Two-stage verification complete!')
print('Compare: Stage 2 should be more fluid, better rhyme/tone than Stage 1.')

In [ ]:
# @title 6. Generate Poetry
%cd /content/poetry-dual-gpt

!python src/sample.py \
    --checkpoint checkpoints/stage2_best.pt \
    --temperature 0.75 \
    --top_p 0.92 \
    --num_samples 3

# Compare Stage 1 vs Stage 2:
# !python src/sample.py --checkpoint checkpoints/stage1_best.pt --num_samples 2

# Interactive (run in separate cell):
# !python src/sample.py --checkpoint checkpoints/stage2_best.pt --interactive

In [ ]:
# @title 7. Save to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = "/content/drive/MyDrive/poetry-dual-gpt/checkpoints"
!mkdir -p {DRIVE_CKPT}

import glob
for ckpt in sorted(glob.glob('checkpoints/stage*_*.pt')):
    !cp -v {ckpt} {DRIVE_CKPT}/
print('✅ Checkpoints saved to Drive')